**Imports**

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [0]:
membership_df=spark.read.table("rental.bike_rental_bronze.membership")

**Is Active**

In [0]:
membership_df = membership_df.withColumn("is_active",F.lit("Y"))

**Membership Days**

In [0]:
membership_df = membership_df.withColumn("membership_days",F.datediff(F.col("end_date"),F.col("start_date")))

**Membership duration bucket**

In [0]:
membership_df = membership_df.withColumn(
    "membership_duration_bucket",
    F.when(F.col("membership_days") <= 5, "short")
     .when(F.col("membership_days") <= 15, "medium")
     .otherwise("long")
)

In [0]:
windowSpec = Window.partitionBy("customer_id").orderBy(F.col("end_date").desc())
membership_df = membership_df.withColumn("rank",F.row_number().over(windowSpec))
membership_df = membership_df.filter(F.col("rank")==1).drop("rank")

In [0]:
membership_df.write.mode("overwrite").saveAsTable("rental.bike_rental_silver.membership")